In [9]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

In [10]:
# Data Ingestion

base_path = Path.cwd()
train_path = base_path / "train"
test_path = base_path / "test"
classes = ["apple", "banana", "orange"]


def make_dataframe(folder):
    files = list(folder.glob("*.jpg"))
    data = []
    for file in files:
        label = file.stem.rsplit("_", 1)[0]
        if label in classes:
            data.append([str(file), label])
    return pd.DataFrame(data, columns=["filename", "class"])

train_df = make_dataframe(train_path).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = make_dataframe(test_path)

print("Training images:", len(train_df))
print("Testing images:", len(test_df))
print(train_df["class"].value_counts())

Training images: 220
Testing images: 55
class
apple     75
banana    73
orange    72
Name: count, dtype: int64


In [11]:
# Preprocessing and Data Augmentation

datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

In [12]:
train_generator = datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    classes=classes,
    subset="training",
    shuffle=True,
    seed=42
)

validation_generator = datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    classes=classes,
    subset="validation",
    shuffle=False,
    seed=42
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    classes=classes,
    shuffle=False
)

Found 176 validated image filenames belonging to 3 classes.
Found 44 validated image filenames belonging to 3 classes.
Found 55 validated image filenames belonging to 3 classes.


In [13]:
# Build the CNN

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(64, 64, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

model.summary()

d:\Advanced Data Science Assignments\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,539 (6.20 MB)

 Trainable params: 1,625,539 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Compile the Model

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
# Train the Model

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

d:\Advanced Data Science Assignments\.venv\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.3523 - loss: 1.2686 - val_accuracy: 0.4318 - val_loss: 0.9961
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.6932 - loss: 0.8640 - val_accuracy: 0.9318 - val_loss: 0.6500
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 435ms/step - accuracy: 0.8352 - loss: 0.5310 - val_accuracy: 0.8864 - val_loss: 0.3837
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 450ms/step - accuracy: 0.8409 - loss: 0.4239 - val_accuracy: 0.8864 - val_loss: 0.3149
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 772ms/step - accuracy: 0.8580 - loss: 0.3830 - val_accuracy: 0.8636 - val_loss: 0.2715
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8580 - loss: 0.2842 - val_accuracy: 0.8636 - val_loss: 0.2901
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 510ms/step - accuracy: 0.9148 - loss: 0.2641 - val_accuracy: 0.9318 - val_loss: 0.2178
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9432 - loss: 0.1655 - val_accuracy: 0.8636 - val_loss: 0.2727
E

In [16]:
# Evaluate the Model

probabilities = model.predict(test_generator)
y_pred = np.argmax(probabilities, axis=1)
y_true = test_generator.classes

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=classes))

2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 895ms/step
Confusion matrix:
[[17  1  1]
 [ 1 16  1]
 [ 0  1 17]]
Classification report:
              precision    recall  f1-score   support

       apple       0.94      0.89      0.92        19
      banana       0.89      0.89      0.89        18
      orange       0.89      0.94      0.92        18

    accuracy                           0.91        55
   macro avg       0.91      0.91      0.91        55
weighted avg       0.91      0.91      0.91        55

